# 06. Sell optimal-stopping model + sequential buy/sell Test

매수 모델과 분리된 매도 모델을 학습한다. 진입 후 완성봉마다 `HOLD/EXIT`를 판단하며 최대 5분에 강제 청산한다.

- Train: OOF buy entries from 2026-07-10 through 2026-07-15
- Validation/policy selection: 2026-07-16
- Final sequential Test: 2026-07-17
- EXIT 체결은 판단 다음 분 `first_bid`
- 동일 종목은 매도 완료 전 새 매수 신호를 무시한다.

In [1]:
from pathlib import Path
import gc
import json
import math
import random
import time

import nbformat
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, balanced_accuracy_score, mean_absolute_error
from torch import nn
from torch.utils.data import DataLoader, Dataset


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


SEED = 20260722
SELL_DATA_VERSION = 'sell_states_conditional_buy_oof_v1'
MODEL_VERSION = 'sell_optimal_stopping_mptsnet_v1'
TRAIN_SESSIONS = [
    'session_2026-07-10', 'session_2026-07-13', 'session_2026-07-14', 'session_2026-07-15',
]
VALID_SESSION = 'session_2026-07-16'
TEST_SESSION = 'session_2026-07-17'
BATCH_SIZE = 256
MAX_EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
ADVANTAGE_LOSS_WEIGHT = 0.35
DOWNSIDE_LOSS_WEIGHT = 0.25
DOWNSIDE_QUANTILE = 0.20
STAKE_USD = 100.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision('high')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PROJECT_ROOT = find_project_root()
PREPROCESS_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
RESULT_ROOT = (PROJECT_ROOT / '../../results/modeling').resolve()
MODEL_ROOT = (PROJECT_ROOT / '../../model').resolve()
sell_schema = json.loads((PREPROCESS_ROOT / f'{SELL_DATA_VERSION}_schema.json').read_text())
state_metadata = pd.read_parquet(PREPROCESS_ROOT / f'{SELL_DATA_VERSION}_metadata.parquet')
with np.load(PREPROCESS_ROOT / f'{SELL_DATA_VERSION}_features.npz') as loaded:
    sequence = loaded['sequence'].astype(np.float32)
    context = loaded['context'].astype(np.float32)

base_notebook = nbformat.read(PROJECT_ROOT / 'notebooks/02_compact_mptsnet_weighted_oof.ipynb', as_version=4)
for cell_index, marker in [(3, 'fit_robust_scaler'), (5, 'CompactMPTSNet')]:
    source = base_notebook.cells[cell_index].source
    assert marker in source, (cell_index, marker)
    exec(compile(source, f'02_base_cell_{cell_index}', 'exec'), globals())

train_indices = np.flatnonzero(
    state_metadata['dataset_split'].eq('oof').to_numpy() & state_metadata['session'].isin(TRAIN_SESSIONS).to_numpy()
)
valid_indices = np.flatnonzero(
    state_metadata['dataset_split'].eq('oof').to_numpy() & state_metadata['session'].eq(VALID_SESSION).to_numpy()
)
test_indices = np.flatnonzero(state_metadata['dataset_split'].eq('test').to_numpy())
print('device:', DEVICE)
print('sequence/context:', sequence.shape, context.shape)
print('train/valid/test states:', len(train_indices), len(valid_indices), len(test_indices))

device: cuda
sequence/context: (9065, 30, 38) (9065, 9)
train/valid/test states: 5475 920 2670


## Sell model

30봉 backbone과 포지션 context를 결합한다. `HOLD` 분류, 미래 최고 EXIT 대비 advantage, HOLD 시 future downside를 함께 예측한다. 데이터가 작으므로 backbone은 32-channel 1-block으로 제한한다.

In [2]:
def fit_context_scaler(values, indices):
    sample = values[indices]
    center = np.median(sample, axis=0).astype(np.float32)
    q25, q75 = np.percentile(sample, [25, 75], axis=0)
    scale = np.maximum((q75 - q25).astype(np.float32), 1e-4)
    return center, scale


sequence_center, sequence_scale = fit_robust_scaler(sequence, train_indices, max_windows=20_000)
sequence_scaled = apply_scaler(sequence, sequence_center, sequence_scale)
context_center, context_scale = fit_context_scaler(context, train_indices)
context_scaled = np.clip((context - context_center) / context_scale, -10, 10).astype(np.float32)
periods = discover_periods(sequence_scaled, train_indices)


class SellDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        index = self.indices[item]
        row = state_metadata.iloc[index]
        return (
            torch.from_numpy(sequence_scaled[index]), torch.from_numpy(context_scaled[index]),
            torch.tensor(float(row['optimal_hold']), dtype=torch.float32),
            torch.tensor(float(row['hold_advantage']) * 100, dtype=torch.float32),
            torch.tensor(float(row['future_downside_from_now']) * 100, dtype=torch.float32), index,
        )


class SellOptimalStoppingModel(nn.Module):
    def __init__(self, input_dim, context_dim, seq_len, periods, d_model=32, dropout=0.20):
        super().__init__()
        encoder = CompactMPTSNet(input_dim, seq_len, periods, d_model=d_model, n_heads=4, blocks=1, dropout=dropout)
        self.input_projection = encoder.input_projection
        self.position = encoder.position
        self.blocks = encoder.blocks
        self.context_encoder = nn.Sequential(
            nn.Linear(context_dim, 32), nn.LayerNorm(32), nn.GELU(), nn.Dropout(dropout), nn.Linear(32, 32),
        )
        self.shared = nn.Sequential(
            nn.LayerNorm(d_model * 3 + 32), nn.Linear(d_model * 3 + 32, 64),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.hold_head = nn.Linear(64, 1)
        self.advantage_head = nn.Linear(64, 1)
        self.downside_head = nn.Linear(64, 1)

    def forward(self, sequence_x, context_x):
        x = self.input_projection(sequence_x) + self.position
        for block in self.blocks:
            x = block(x)
        pooled = torch.cat([x.mean(dim=1), x.amax(dim=1), x[:, -1]], dim=1)
        shared = self.shared(torch.cat([pooled, self.context_encoder(context_x)], dim=1))
        return {
            'hold_logit': self.hold_head(shared).squeeze(1),
            'advantage_pct': self.advantage_head(shared).squeeze(1),
            'downside_pct': self.downside_head(shared).squeeze(1),
        }


def pinball_loss(prediction, target, quantile):
    error = target - prediction
    return torch.maximum(quantile * error, (quantile - 1) * error).mean()


def spearman(actual, predicted):
    # 위치 기준 비교: metadata의 원래 index와 예측 배열 index가 정렬되지 않게 한다.
    actual = np.asarray(actual, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    value = pd.Series(actual).corr(pd.Series(predicted), method='spearman')
    return 0.0 if pd.isna(value) else float(value)

print('periods:', periods)

periods: [15, 10, 8]


## Train과 validation policy 선택

모델 checkpoint는 validation HOLD PR-AUC와 advantage/downside 순위상관으로 선택한다. 실제 HOLD 확률·advantage threshold·downside penalty는 validation sequential return으로 고른다.

In [3]:
@torch.no_grad()
def predict_states(model, indices):
    loader = DataLoader(SellDataset(indices), batch_size=BATCH_SIZE, shuffle=False)
    model.eval()
    result = {'state_index': [], 'hold_probability': [], 'predicted_advantage': [], 'predicted_downside': []}
    for batch_sequence, batch_context, _, _, _, batch_indices in loader:
        output = model(batch_sequence.to(DEVICE), batch_context.to(DEVICE))
        result['state_index'].append(batch_indices.numpy())
        result['hold_probability'].append(torch.sigmoid(output['hold_logit']).cpu().numpy())
        result['predicted_advantage'].append(output['advantage_pct'].cpu().numpy() / 100)
        result['predicted_downside'].append(output['downside_pct'].cpu().numpy() / 100)
    return {key: np.concatenate(value) for key, value in result.items()}


def state_metrics(prediction):
    rows = state_metadata.iloc[prediction['state_index']]
    target = rows['optimal_hold'].to_numpy()
    probability = prediction['hold_probability']
    return {
        'hold_prevalence': float(target.mean()),
        'hold_pr_auc': float(average_precision_score(target, probability)),
        'hold_balanced_accuracy': float(balanced_accuracy_score(target, probability >= 0.5)),
        'advantage_mae': float(mean_absolute_error(rows['hold_advantage'], prediction['predicted_advantage'])),
        'advantage_spearman': spearman(rows['hold_advantage'], prediction['predicted_advantage']),
        'downside_mae': float(mean_absolute_error(rows['future_downside_from_now'], prediction['predicted_downside'])),
        'downside_spearman': spearman(rows['future_downside_from_now'], prediction['predicted_downside']),
    }


train_hold = state_metadata.iloc[train_indices]['optimal_hold'].to_numpy(np.float32)
pos_weight_value = float((len(train_hold) - train_hold.sum()) / max(train_hold.sum(), 1))
model = SellOptimalStoppingModel(sequence.shape[-1], context.shape[-1], sequence.shape[1], periods).to(DEVICE)
hold_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, device=DEVICE))
advantage_criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=3e-5)
loader = DataLoader(
    SellDataset(train_indices), batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(SEED), pin_memory=DEVICE.type == 'cuda',
)
best_score, best_state, best_epoch, stale = -np.inf, None, 0, 0
history = []
started = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    totals = {'loss': 0.0, 'hold': 0.0, 'advantage': 0.0, 'downside': 0.0}
    for batch_sequence, batch_context, batch_hold, batch_advantage, batch_downside, _ in loader:
        batch_sequence = batch_sequence.to(DEVICE, non_blocking=True)
        batch_context = batch_context.to(DEVICE, non_blocking=True)
        batch_hold = batch_hold.to(DEVICE, non_blocking=True)
        batch_advantage = batch_advantage.to(DEVICE, non_blocking=True)
        batch_downside = batch_downside.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        output = model(batch_sequence, batch_context)
        hold_loss = hold_criterion(output['hold_logit'], batch_hold)
        advantage_loss = advantage_criterion(output['advantage_pct'], batch_advantage)
        downside_loss = pinball_loss(output['downside_pct'], batch_downside, DOWNSIDE_QUANTILE)
        loss = hold_loss + ADVANTAGE_LOSS_WEIGHT * advantage_loss + DOWNSIDE_LOSS_WEIGHT * downside_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        for key, value in [('loss', loss), ('hold', hold_loss), ('advantage', advantage_loss), ('downside', downside_loss)]:
            totals[key] += value.item() * len(batch_sequence)
    valid_prediction = predict_states(model, valid_indices)
    metrics = state_metrics(valid_prediction)
    selection_score = metrics['hold_pr_auc'] + 0.25 * metrics['advantage_spearman'] + 0.10 * metrics['downside_spearman']
    if not np.isfinite(selection_score):
        selection_score = -1e9
    history.append({
        'epoch': epoch, **{f'train_{key}_loss': value / len(train_indices) for key, value in totals.items()},
        **{f'valid_{key}': value for key, value in metrics.items()}, 'selection_score': selection_score,
    })
    if best_state is None or selection_score > best_score + 1e-4:
        best_score = selection_score
        best_epoch = epoch
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        stale = 0
    else:
        stale += 1
    scheduler.step()
    if stale >= PATIENCE:
        break
model.load_state_dict(best_state)
training_history = pd.DataFrame(history)
train_prediction = predict_states(model, train_indices)
valid_prediction = predict_states(model, valid_indices)
print(f'training elapsed: {(time.time() - started) / 60:.2f} min | best epoch={best_epoch}')
print('train:', state_metrics(train_prediction))
print('valid:', state_metrics(valid_prediction))

/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


training elapsed: 0.58 min | best epoch=28
train: {'hold_prevalence': 0.4390867579908676, 'hold_pr_auc': 0.7093239131274558, 'hold_balanced_accuracy': 0.7101443052418335, 'advantage_mae': 0.015594554677488624, 'advantage_spearman': 0.33170696680024947, 'downside_mae': 0.02050537241352532, 'downside_spearman': 0.28859024566100555}
valid: {'hold_prevalence': 0.41956521739130437, 'hold_pr_auc': 0.6454539514674354, 'hold_balanced_accuracy': 0.6974976227901651, 'advantage_mae': 0.011895760604094549, 'advantage_spearman': 0.2658288064355055, 'downside_mae': 0.014953443461895997, 'downside_spearman': 0.30516036253526474}


In [4]:
def attach_prediction(prediction):
    frame = state_metadata.iloc[prediction['state_index']].copy().reset_index(drop=True)
    for key in ['hold_probability', 'predicted_advantage', 'predicted_downside']:
        frame[key] = prediction[key]
    return frame


def choose_exit_states(prediction_frame, probability_threshold, advantage_threshold, downside_penalty):
    selected = []
    for _, group in prediction_frame.sort_values('hold_minute').groupby('entry_id', sort=False):
        chosen = None
        for _, row in group.sort_values('hold_minute').iterrows():
            risk_adjusted_advantage = row['predicted_advantage'] - downside_penalty * max(-row['predicted_downside'], 0)
            hold = (
                row['hold_minute'] < sell_schema['max_hold_minutes']
                and row['hold_probability'] >= probability_threshold
                and risk_adjusted_advantage > advantage_threshold
            )
            if not hold:
                chosen = row.copy()
                break
        if chosen is None:
            chosen = group.sort_values('hold_minute').iloc[-1].copy()
        selected.append(chosen)
    return pd.DataFrame(selected)


def sequential_filter(exit_choices):
    selected = []
    for _, group in exit_choices.sort_values('entry_timestamp').groupby(['session', 'symbol'], sort=False):
        available_at = None
        for index, row in group.sort_values('entry_timestamp').iterrows():
            if available_at is None or row['entry_timestamp'] >= available_at:
                selected.append(index)
                available_at = row['exit_execution_timestamp']
    return exit_choices.loc[selected].sort_values('entry_timestamp').copy()


def return_metrics(trades):
    returns = trades['current_net_return'].to_numpy()
    pnl = returns * STAKE_USD
    equity = np.cumsum(pnl)
    peak = np.maximum.accumulate(np.r_[0.0, equity])
    gross_profit = pnl[pnl > 0].sum()
    gross_loss = -pnl[pnl < 0].sum()
    return {
        'trades': int(len(trades)), 'win_rate': float((returns > 0).mean()),
        'mean_return': float(returns.mean()), 'median_return': float(np.median(returns)),
        'risk_adjusted_return': float(returns.mean() - returns.std(ddof=1) / math.sqrt(len(returns))) if len(returns) > 1 else float(returns.mean()),
        'total_pnl_usd': float(pnl.sum()),
        'profit_factor': float(gross_profit / gross_loss) if gross_loss > 0 else float('inf'),
        'max_drawdown_usd': float((np.r_[0.0, equity] - peak).min()),
        'mean_hold_minutes': float(trades['hold_minute'].mean()),
    }


valid_frame = attach_prediction(valid_prediction)
policy_rows = []
for probability_threshold in [0.40, 0.50, 0.60, 0.70]:
    for advantage_threshold in [-0.002, 0.0, 0.002]:
        for downside_penalty in [0.0, 0.25, 0.50, 1.0]:
            choices = choose_exit_states(valid_frame, probability_threshold, advantage_threshold, downside_penalty)
            policy_rows.append({
                'probability_threshold': probability_threshold, 'advantage_threshold': advantage_threshold,
                'downside_penalty': downside_penalty, **return_metrics(choices),
            })
policy_table = pd.DataFrame(policy_rows)
eligible = policy_table[policy_table['trades'].ge(50)]
selected_policy = eligible.sort_values(['risk_adjusted_return', 'total_pnl_usd']).iloc[-1]
print('selected policy')
display(selected_policy.to_frame('value'))
display(policy_table.sort_values(['risk_adjusted_return', 'total_pnl_usd'], ascending=False).head(15))

selected policy


,value
probability_threshold,0.400000
advantage_threshold,0.000000
downside_penalty,0.250000
trades,184.000000
win_rate,0.347826
mean_return,-0.003699
median_return,-0.005945
risk_adjusted_return,-0.005714
total_pnl_usd,-68.068885
profit_factor,0.673469


,probability_threshold,advantage_threshold,downside_penalty,trades,win_rate,mean_return,median_return,risk_adjusted_return,total_pnl_usd,profit_factor,max_drawdown_usd,mean_hold_minutes
5,0.4,0.000,0.25,184,0.347826,-0.003699,-0.005945,-0.005714,-68.068885,0.673469,-68.068885,2.815217
29,0.6,0.000,0.25,184,0.326087,-0.003914,-0.006045,-0.005859,-72.009224,0.640561,-72.009224,2.516304
25,0.6,-0.002,0.25,184,0.320652,-0.003662,-0.006954,-0.005868,-67.386605,0.688480,-78.792771,2.880435
33,0.6,0.002,0.25,184,0.309783,-0.004732,-0.006052,-0.006236,-87.063728,0.525011,-87.063728,1.951087
17,0.5,0.000,0.25,184,0.331522,-0.004274,-0.006335,-0.006248,-78.635020,0.625904,-78.635020,2.750000
2,0.4,-0.002,0.50,184,0.315217,-0.004929,-0.006335,-0.006372,-90.691756,0.513293,-90.691756,1.880435
26,0.6,-0.002,0.50,184,0.304348,-0.005029,-0.005900,-0.006395,-92.534432,0.484431,-92.534432,1.739130
38,0.7,-0.002,0.50,184,0.293478,-0.005181,-0.005892,-0.006521,-95.323660,0.460554,-95.323660,1.461957
21,0.5,0.002,0.25,184,0.304348,-0.005042,-0.006335,-0.006557,-92.763689,0.506533,-92.763689,1.972826
9,0.4,0.002,0.25,184,0.304348,-0.005083,-0.006335,-0.006601,-93.526137,0.504487,-93.526137,1.978261


## Final sequential Test

Test의 모든 buy signal에 매도 모델을 적용한 뒤, 같은 종목에서 기존 포지션이 청산되기 전 발생한 매수는 제거한다. Test 결과는 정책 선택에 사용하지 않는다.

In [5]:
test_prediction = predict_states(model, test_indices)
test_state_metrics = state_metrics(test_prediction)
test_frame = attach_prediction(test_prediction)
test_choices = choose_exit_states(
    test_frame, float(selected_policy['probability_threshold']),
    float(selected_policy['advantage_threshold']), float(selected_policy['downside_penalty']),
)
test_trades = sequential_filter(test_choices)
test_metrics = return_metrics(test_trades)

def benchmark_choices(frame, mode):
    groups = frame.groupby('entry_id', sort=False)
    if mode == 'exit_1m':
        return groups.apply(lambda g: g.loc[g['hold_minute'].idxmin()], include_groups=False).reset_index(drop=True)
    if mode == 'exit_5m':
        return groups.apply(lambda g: g.loc[g['hold_minute'].idxmax()], include_groups=False).reset_index(drop=True)
    if mode == 'oracle':
        return groups.apply(lambda g: g.loc[g['current_net_return'].idxmax()], include_groups=False).reset_index(drop=True)
    raise ValueError(mode)

comparison_rows = [{'policy': 'AI sell', **test_metrics}]
for mode in ['exit_1m', 'exit_5m', 'oracle']:
    choices = benchmark_choices(test_frame, mode)
    comparison_rows.append({'policy': mode, **return_metrics(sequential_filter(choices))})
comparison = pd.DataFrame(comparison_rows)

# 청산 품질 진단용 동일-entry 비교다. AI 청산으로 실행된 entry만 고정하므로
# fixed 1m/5m 결과는 독립적인 sequential 전략 성과가 아니라 counterfactual이다.
matched_frame = test_frame[test_frame['entry_id'].isin(test_trades['entry_id'])].copy()
matched_rows = [{'policy': 'AI sell', **return_metrics(test_trades)}]
for mode in ['exit_1m', 'exit_5m', 'oracle']:
    matched_rows.append({'policy': mode, **return_metrics(benchmark_choices(matched_frame, mode))})
matched_comparison = pd.DataFrame(matched_rows)
raw_candidate_oracle = return_metrics(benchmark_choices(test_frame, 'oracle'))
valid_choices = choose_exit_states(
    valid_frame, float(selected_policy['probability_threshold']),
    float(selected_policy['advantage_threshold']), float(selected_policy['downside_penalty']),
)
valid_metrics = return_metrics(valid_choices)
DEPLOYMENT_ELIGIBLE = bool(
    valid_metrics['risk_adjusted_return'] > 0 and valid_metrics['profit_factor'] > 1
    and test_metrics['risk_adjusted_return'] > 0 and test_metrics['profit_factor'] > 1
    and test_metrics['trades'] >= 20
)

model_path = MODEL_ROOT / f'{MODEL_VERSION}.pt'
metrics_path = RESULT_ROOT / f'{MODEL_VERSION}_metrics.json'
torch.save({
    'model_version': MODEL_VERSION, 'architecture': 'SellOptimalStoppingModel',
    'state_dict': best_state, 'sequence_input_dim': sequence.shape[-1],
    'context_names': sell_schema['context_names'], 'periods': periods,
    'sequence_center': torch.from_numpy(sequence_center.copy()),
    'sequence_scale': torch.from_numpy(sequence_scale.copy()),
    'context_center': torch.from_numpy(context_center.copy()),
    'context_scale': torch.from_numpy(context_scale.copy()),
    'best_epoch': best_epoch, 'selected_policy': selected_policy.to_dict(),
    'deployment_eligible': DEPLOYMENT_ELIGIBLE,
}, model_path)
training_history.to_parquet(RESULT_ROOT / f'{MODEL_VERSION}_training_history.parquet', index=False, compression='zstd')
policy_table.to_parquet(RESULT_ROOT / f'{MODEL_VERSION}_validation_policies.parquet', index=False, compression='zstd')
test_frame.to_parquet(RESULT_ROOT / f'{MODEL_VERSION}_test_state_predictions.parquet', index=False, compression='zstd')
test_trades.to_parquet(RESULT_ROOT / f'{MODEL_VERSION}_sequential_test_trades.parquet', index=False, compression='zstd')
comparison.to_parquet(RESULT_ROOT / f'{MODEL_VERSION}_test_comparison.parquet', index=False, compression='zstd')
matched_comparison.to_parquet(RESULT_ROOT / f'{MODEL_VERSION}_matched_exit_comparison.parquet', index=False, compression='zstd')
metrics = {
    'model_version': MODEL_VERSION, 'parameters': sum(p.numel() for p in model.parameters()),
    'train_sessions': TRAIN_SESSIONS, 'validation_session': VALID_SESSION, 'test_session': TEST_SESSION,
    'best_epoch': best_epoch, 'periods': periods, 'state_metrics_test': test_state_metrics,
    'selected_policy': {key: float(value) for key, value in selected_policy.to_dict().items()},
    'validation_sequential': valid_metrics, 'test_sequential': test_metrics,
    'test_comparison': comparison.to_dict(orient='records'),
    'matched_exit_comparison': matched_comparison.to_dict(orient='records'),
    'raw_candidate_oracle_ceiling': raw_candidate_oracle,
    'deployment_eligible': DEPLOYMENT_ELIGIBLE,
}
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print('deployment eligible:', DEPLOYMENT_ELIGIBLE)
print('model:', model_path)
display(comparison)
display(matched_comparison)
display(test_trades[[
    'entry_timestamp', 'exit_execution_timestamp', 'symbol', 'hold_minute',
    'current_net_return', 'hold_probability', 'predicted_advantage', 'predicted_downside',
]])

deployment eligible: False
model: /home/user/urbandatalab/YSLee/model/sell_optimal_stopping_mptsnet_v1.pt


,policy,trades,win_rate,mean_return,median_return,risk_adjusted_return,total_pnl_usd,profit_factor,max_drawdown_usd,mean_hold_minutes
0,AI sell,318,0.283019,-0.009818,-0.008964,-0.011403,-312.200790,0.364076,-317.878425,2.097484
1,exit_1m,534,0.277154,-0.008765,-0.008293,-0.009676,-468.039088,0.306854,-473.356586,1.000000
2,exit_5m,175,0.302857,-0.015033,-0.011478,-0.017935,-263.074353,0.308199,-274.465531,5.000000
3,oracle,303,0.293729,-0.004999,-0.008219,-0.006460,-151.459711,0.584963,-170.840762,2.389439


,policy,trades,win_rate,mean_return,median_return,risk_adjusted_return,total_pnl_usd,profit_factor,max_drawdown_usd,mean_hold_minutes
0,AI sell,318,0.283019,-0.009818,-0.008964,-0.011403,-312.200790,0.364076,-317.878425,2.097484
1,exit_1m,318,0.283019,-0.008689,-0.008964,-0.009899,-276.308904,0.328464,-283.978934,1.000000
2,exit_5m,318,0.279874,-0.016856,-0.014050,-0.019131,-536.010884,0.294290,-548.421554,5.000000
3,oracle,318,0.503145,0.006677,0.002136,0.005058,212.330836,1.944826,-53.879742,2.515723


,entry_timestamp,exit_execution_timestamp,symbol,hold_minute,current_net_return,hold_probability,predicted_advantage,predicted_downside
3,2026-07-17 08:50:00+00:00,2026-07-17 08:54:00+00:00,CJMB,4,-0.023048,0.387827,0.000870,-0.006345
17,2026-07-17 08:55:00+00:00,2026-07-17 08:58:00+00:00,CJMB,3,0.015474,0.480512,0.000656,-0.008590
31,2026-07-17 09:03:00+00:00,2026-07-17 09:05:00+00:00,CJMB,2,0.007846,0.583543,0.002918,-0.011995
26,2026-07-17 09:03:00+00:00,2026-07-17 09:05:00+00:00,GCTK,2,-0.000138,0.571056,0.003788,-0.024760
35,2026-07-17 09:13:00+00:00,2026-07-17 09:14:00+00:00,GCTK,1,-0.005021,0.701377,0.007431,-0.035132
...,...,...,...,...,...,...,...,...
2645,2026-07-17 22:39:00+00:00,2026-07-17 22:40:00+00:00,STAK,1,-0.005315,0.573024,0.002071,-0.011245
2650,2026-07-17 22:40:00+00:00,2026-07-17 22:41:00+00:00,STAK,1,-0.005315,0.540822,0.001942,-0.011964
2655,2026-07-17 22:43:00+00:00,2026-07-17 22:44:00+00:00,STAK,1,-0.005315,0.534601,0.001587,-0.010006
2660,2026-07-17 22:45:00+00:00,2026-07-17 22:46:00+00:00,STAK,1,-0.005315,0.557725,0.002395,-0.010782
